# 04 — Target creation (Lending Club `default_risk`)

Build a **binary** supervision label from `loan_status` on the featured extract, drop ambiguous rows, summarize imbalance, and write a model-ready CSV.

**Input:** `datasets/processed/lendingclub_featured.csv`  
**Output:** `datasets/processed/lendingclub_model_ready.csv`

Libraries: **pandas**, **matplotlib** only.

## 1. Load data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for directory in (cwd, *cwd.parents):
        if (directory / "backend").is_dir() and (directory / "ml").is_dir():
            return directory
    return cwd


def resolve_featured_csv(processed_dir: Path) -> Path:
    if not processed_dir.is_dir():
        raise FileNotFoundError(
            f"Processed directory not found: {processed_dir.resolve()}"
        )
    canonical = processed_dir / "lendingclub_featured.csv"
    if canonical.is_file():
        return canonical.resolve()
    candidates = sorted(
        p.resolve()
        for p in processed_dir.glob("*.csv")
        if "featured" in p.name.lower() and "lendingclub" in p.name.lower()
    )
    if not candidates:
        listing = sorted(p.name for p in processed_dir.glob("*.csv"))
        raise FileNotFoundError(
            f"No featured Lending Club CSV in {processed_dir.resolve()}. "
            f"Expected lendingclub_featured.csv. CSV files present: {listing or '(none)'}"
        )
    return candidates[0]


REPO_ROOT = find_repo_root()
PROC = REPO_ROOT / "datasets" / "processed"
DATA_PATH = resolve_featured_csv(PROC)

df = pd.read_csv(DATA_PATH, low_memory=False)
mem = int(df.memory_usage(deep=True).sum())

print(f"Resolved: {DATA_PATH}")
print(f"Shape: {df.shape}")
print(f"Memory (deep estimate): {mem / (1024 ** 2):.2f} MiB ({mem:,} bytes)")

## 2. Inspect `loan_status`

In [ ]:
if "loan_status" not in df.columns:
    raise KeyError(
        "Column 'loan_status' is required for target creation. "
        f"Columns: {list(df.columns)[:40]}..."
    )

status = df["loan_status"].astype(str).str.strip()
unique_status = sorted(status.dropna().unique())
print(f"Unique loan_status values ({len(unique_status)}):")
for v in unique_status:
    print(f"  - {v!r}")

print("\nValue counts:")
print(df["loan_status"].astype(str).str.strip().value_counts().to_string())

## 3. Map outcomes → `default_risk` (binary)

- **Safe (0):** `Fully Paid`, `Current`
- **Risky (1):** `Charged Off`, `Default`, `Late (31-120 days)`, `Late (16-30 days)`
- **Excluded (rows dropped):** statuses containing **In Grace Period**, **Issued**, or **Does not meet credit policy** (covers LC variants such as *Status:Fully Paid* suffixes)
- **Other unknown labels:** dropped (not mapped to Safe/Risky)

In [ ]:
SAFE_LABELS = {
    "Fully Paid",
    "Current",
}

RISKY_LABELS = {
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)",
}

AMBIGUOUS_SUBSTRINGS = (
    "In Grace Period",
    "Issued",
    "Does not meet credit policy",
    "Does not meet the credit policy",
)

s = df["loan_status"].astype(str).str.strip()


def is_ambiguous(val: str) -> bool:
    v = str(val).strip()
    if v.lower() in {"nan", "none"}:
        return True
    return any(sub in v for sub in AMBIGUOUS_SUBSTRINGS)


ambiguous_mask = s.map(is_ambiguous)
n_ambiguous = int(ambiguous_mask.sum())
print(f"Rows flagged ambiguous / invalid loan_status: {n_ambiguous:,}")

df_filtered = df.loc[~ambiguous_mask].copy()
s2 = df_filtered["loan_status"].astype(str).str.strip()

in_safe = s2.isin(SAFE_LABELS)
in_risky = s2.isin(RISKY_LABELS)
mapped = in_safe | in_risky
n_unmapped = int((~mapped).sum())
if n_unmapped:
    print(f"Dropping {n_unmapped:,} rows with unmapped loan_status (after ambiguous exclusion):")
    print(s2.loc[~mapped].value_counts().head(20))

df_model = df_filtered.loc[mapped].copy()
s3 = df_model["loan_status"].astype(str).str.strip()

df_model["default_risk"] = s3.map(lambda x: 0 if x in SAFE_LABELS else 1).astype("int8")

print(f"\nRows retained for modeling: {len(df_model):,}")

## 4. Class distribution

In [ ]:
vc = df_model["default_risk"].value_counts().sort_index()
pct = df_model["default_risk"].value_counts(normalize=True).sort_index() * 100

dist = pd.DataFrame({"count": vc, "percent": pct.round(4)})
dist.index = dist.index.map({0: "Safe (0)", 1: "Risky (1)"})
print("default_risk distribution:")
print(dist.to_string())

print("\nPercentages (Risky share):")
print(f"  Risky (1): {pct.get(1, 0):.4f}%")

## 5. Class imbalance plot

In [ ]:
labels = ["Safe (0)", "Risky (1)"]
counts = [int(vc.get(0, 0)), int(vc.get(1, 0))]
colors = ["#2ecc71", "#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(labels, counts, color=colors, edgecolor="black")
axes[0].set_title("Class counts")
axes[0].set_ylabel("Rows")
for i, c in enumerate(counts):
    axes[0].text(i, c, f"{c:,}", ha="center", va="bottom", fontsize=10)

pcts = [pct.get(0, 0), pct.get(1, 0)]
axes[1].bar(labels, pcts, color=colors, edgecolor="black")
axes[1].set_title("Class share (%)")
axes[1].set_ylabel("Percent")
axes[1].set_ylim(0, max(pcts) * 1.25 if max(pcts) else 1)
for i, p in enumerate(pcts):
    axes[1].text(i, p, f"{p:.2f}%", ha="center", va="bottom", fontsize=10)

fig.suptitle("default_risk class imbalance", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Build model-ready frame & save

All feature columns from the filtered rows (everything except `loan_status`), plus **`default_risk`** as the last column.

In [ ]:
feature_cols = [c for c in df_model.columns if c not in {"loan_status", "default_risk"}]
out = df_model[feature_cols + ["default_risk"]].copy()

print(f"Output shape: {out.shape}")
print(f"Features: {len(feature_cols)}  |  target: default_risk")

OUTPUT_PATH = PROC / "lendingclub_model_ready.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f"\nWrote: {OUTPUT_PATH.resolve()}")